# 📊 Dataset Palmer Penguins

## 1. Carregamento dos Dados



Vamos trabalhar com o dataset **Palmer Penguins**, já explorado em aulas anteriores.  

In [ ]:
!pip install palmerpenguins --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score, f1_score
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

from palmerpenguins import load_penguins

Como já realizamos a etapa de **Análise Exploratória de Dados (EDA)** anteriormente, iremos avançar diretamente para a aplicação dos modelos de Machine Learning.  

O código abaixo já contempla todas as etapas de **transformação do DataFrame** que foram aplicadas anteriormente, deixando os dados prontos para o treinamento dos modelos.  


In [ ]:
df = load_penguins()

In [ ]:
df.shape

In [ ]:
df.dropna(inplace=True) # Remover todas as linhas com valores ausentes

In [ ]:
def remove_outliers_iqr(df, column, category_col=None):
    if category_col:
        filtered_df = pd.DataFrame()
        for category in df[category_col].unique():
            subset = df[df[category_col] == category]
            Q1 = subset[column].quantile(0.25)
            Q3 = subset[column].quantile(0.75)
            IQR = Q3 - Q1
            mask = (subset[column] >= Q1 - 1.5 * IQR) & (subset[column] <= Q3 + 1.5 * IQR)
            filtered_df = pd.concat([filtered_df, subset[mask]])
        return filtered_df
    else:
        Q1 = df[column].quantile(0.25)
        Q3 = df[column].quantile(0.75)
        IQR = Q3 - Q1
        mask = (df[column] >= Q1 - 1.5 * IQR) & (df[column] <= Q3 + 1.5 * IQR)
        return df[mask]

In [ ]:
X = df.drop(columns=['species']).copy()
y = df['species'].copy()

In [ ]:
categorical = X.select_dtypes(include='object').columns.tolist()
categorical

In [ ]:
numerical = X.select_dtypes(include='number').columns.tolist()
numerical

In [ ]:
for column in numerical:
  df = remove_outliers_iqr(df, column, 'species')

In [ ]:
df.shape

In [ ]:
df.head(10)

In [ ]:
X = df.drop(columns=['species']).copy()
y = df['species'].copy()

In [ ]:
df_penguins = df.copy()

## Pré-processamento

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
categorical = X_train.select_dtypes(include='object').columns.tolist()
categorical

In [ ]:
numerical = X_train.select_dtypes(include='number').columns.tolist()
numerical

# **📌 Naive Bayes**

**Naive Bayes** é uma família de classificadores probabilísticos baseados no **Teorema de Bayes**.  
Eles fazem a hipótese "ingênua" de que as features são **independentes entre si**, o que raramente é 100% verdadeiro, mas costuma funcionar muito bem na prática.

**O que vale para todos os Naive Bayes:**
- Baseados no **Teorema de Bayes**.  
- Supõem **independência condicional** entre as features.  
- São **simples, rápidos e eficientes**, mesmo em datasets grandes e de alta dimensionalidade.  
- Têm **boa capacidade de generalização** mesmo quando as suposições não são totalmente satisfeitas.  
- São muito usados em **texto** (ex: filtragem de spam, análise de sentimentos, categorização de documentos).  

> Variações do Naive Bayes

➡️  **GaussianNB**  
  - Para dados **contínuos**.  
  - Assume que cada atributo segue uma **distribuição normal (gaussiana)** dentro de cada classe.  
  - Estima **média e variância** de cada feature por classe.  
  - Exemplo: medidas numéricas no dataset **Iris**.  

➡️  **MultinomialNB**  
  - Para dados de **contagem ou frequência** (inteiros não-negativos).  
  - Muito usado em **texto**, com `CountVectorizer` ou `TfidfVectorizer`.  
  - Exemplo: número de vezes que cada palavra aparece em um e-mail (detecção de spam).  

➡️  **BernoulliNB**  
  - Para dados **binários (0/1)**.  
  - Indica apenas **presença ou ausência** de uma feature.  
  - Exemplo: classificação de textos considerando só se uma palavra aparece ou não.  

💡 **Resumo rápido:**  
- **GaussianNB** → atributos contínuos.  
- **MultinomialNB** → contagens/frequências.  
- **BernoulliNB** → atributos binários.


## Treinamento do modelo `GaussianNB`  

> **🎯 Momento de Praticar**

In [ ]:
pre_gnb = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical)
    ]
)

O Gaussian Naive Bayes assume que as **variáveis são contínuas e seguem uma distribuição normal (gaussiana)**. Quando você aplica um `OneHotEncoder`, os dados categóricos viram variáveis binárias (0 ou 1), e essas variáveis **não seguem uma distribuição normal**, o que **viola a suposição central** do GNB.

In [ ]:
pre_gnb = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical),
        # ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical)
    ],
    # remainder = 'drop'  (default) -> poderia ser 'passthrough'
)

⚠️ O `StandardScaler()` só vai ser aplicado às colunas listadas em `numerical`. E por padrão, todas as outras colunas são descartadas.

In [ ]:
pipe_gnb = Pipeline(steps=[
    ('pre', pre_gnb),
    ('gnb', GaussianNB())
])

In [ ]:
pipe_gnb.fit(X_train, y_train)

In [ ]:
y_pred = pipe_gnb.predict(X_test)

## Avaliação dos resultados

In [ ]:
print(classification_report(y_test, y_pred, target_names=pipe_gnb.classes_))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=pipe_gnb.classes_)

disp.plot(cmap='Blues')
plt.title("Matriz de Confusão")
plt.tight_layout()
plt.show()

In [ ]:
cv_acc_gnb = cross_val_score(pipe_gnb, X, y, cv=10, scoring='accuracy')

print(f"Acurácias da Validação Cruzada: {cv_acc_gnb}")
print(f"Acurácia Média CV: {np.mean(cv_acc_gnb):.4f}")
print(f"Desvio Padrão CV: {np.std(cv_acc_gnb):.4f}")

In [ ]:
cv_f1_gnb = cross_val_score(pipe_gnb, X, y, cv=10, scoring='f1_macro')

print(f"F1 da Validação Cruzada: {cv_f1_gnb}")
print(f"F1 Média CV: {np.mean(cv_f1_gnb):.4f}")
print(f"Desvio Padrão CV: {np.std(cv_f1_gnb):.4f}")

# **📌 Support Vector Machine (SVM)**

O **Support Vector Machine (SVM)** é um algoritmo de aprendizado supervisionado que pertence à classe dos **métodos baseados em margem**. Diferente de métodos baseados em instância como o KNN, o SVM **constrói um modelo durante o treinamento**, buscando encontrar um **hiperplano ótimo** que separa as classes com a **maior margem possível** entre os exemplos de classes diferentes.

Na **classificação**, o objetivo do SVM é maximizar essa margem, ou seja, a distância entre o hiperplano de separação e os exemplos mais próximos de cada classe, chamados de **vetores de suporte**. Esses vetores são os pontos mais críticos do conjunto de dados, pois definem a posição do hiperplano.

O SVM é especialmente poderoso em **problemas de alta dimensionalidade** e pode lidar com **relações não lineares** por meio do uso de **funções kernel**, que transformam os dados para um espaço de maior dimensão onde se torna possível encontrar uma separação linear.

Embora o SVM possa apresentar ótimo desempenho em muitos cenários, ele pode ser sensível à escolha dos hiperparâmetros (como o tipo de kernel e o parâmetro de regularização) e pode ter um custo computacional elevado em conjuntos de dados muito grandes.

### 💡 Exemplo didático

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, make_moons
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

np.random.seed(42)

# Dataset quase linear, pequeno
# Cria clusters gaussianos, geralmente quase lineares.
X_blobs, y_blobs = make_blobs(n_samples=80, centers=2, cluster_std=1.3, random_state=42)

# Dataset não linear, pequeno
# Cria duas “meias-luas” entrelaçadas, não linear.c
X_moons, y_moons = make_moons(n_samples=120, noise=0.20, random_state=42)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(X_blobs[:,0], X_blobs[:,1], c=y_blobs, cmap="bwr", alpha=0.7)
axes[1].scatter(X_moons[:,0], X_moons[:,1], c=y_moons, cmap="bwr", alpha=0.7)

plt.title("Dados artificiais SVM")
plt.show()

Função para treinar e plotar fronteira

In [ ]:
def fit_and_plot(ax, X, y, kernel='linear', C=1.0, gamma='scale', title=''):
    clf = Pipeline([
        ('scaler', StandardScaler()),
        ('svc', SVC(kernel=kernel, C=C, gamma=gamma))
    ])
    clf.fit(X, y)

    # grade mais leve
    x_min, x_max = X[:,0].min() - 1.0, X[:,0].max() + 1.0
    y_min, y_max = X[:,1].min() - 1.0, X[:,1].max() + 1.0
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                         np.linspace(y_min, y_max, 200))
    Z = clf.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    # regiões e margem
    ax.contourf(xx, yy, Z > 0, alpha=0.2)
    ax.contour(xx, yy, Z, levels=[-1, 0, 1], linestyles=['--','-','--'], linewidths=1)

    # dados
    ax.scatter(X[:,0], X[:,1], c=y, s=30, edgecolor='k')

    # vetores de suporte
    sv_idx = clf.named_steps['svc'].support_
    ax.scatter(X[sv_idx,0], X[sv_idx,1], s=100, facecolors='none', edgecolors='k', linewidths=1.5)

    # metricas somente de treino
    y_pred = clf.predict(X)
    acc = accuracy_score(y, y_pred)

    n_sv = clf.named_steps['svc'].n_support_.sum()
    ax.set_title(f"{title}\nacc treino={acc:.2f} | #SV={n_sv}")
    ax.set_xlabel("x1"); ax.set_ylabel("x2")


> Linear SVM com C pequeno vs C grande (dados quase lineares)

- O SVM busca uma fronteira que maximize a margem entre as classes.
- O hiperparâmetro **C** controla a penalidade por erros no treino.
- **C pequeno**: mais tolerância a erros, margem mais larga e modelo mais simples.
- **C grande**: pouca tolerância a erros, margem mais estreita e maior risco de superajuste.
- Compare nos gráficos a largura da margem, os pontos próximos à fronteira e a quantidade de erros no treino.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

fit_and_plot(axes[0], X_blobs, y_blobs, kernel='linear', C=0.1, title="Linear • C=0.01")
fit_and_plot(axes[1], X_blobs, y_blobs, kernel='linear', C=100, title="Linear • C=100")

plt.tight_layout(); plt.show()


> RBF SVM com γ baixo vs γ alto (dados não lineares)

- O kernel RBF permite fronteiras **curvas** para separar dados não lineares.
- O hiperparâmetro **γ** controla o alcance da influência de cada ponto.
- **γ baixo**: fronteiras mais suaves e globais, menor risco de superajuste.
- **γ alto**: fronteiras muito onduladas e locais, maior risco de superajuste.
- Observe como a fronteira muda quando o γ aumenta e como isso afeta a separação visual das classes.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

fit_and_plot(axes[0], X_moons, y_moons, kernel='rbf', C=1.0, gamma=0.3, title="RBF • C=1 • γ=0.3")
fit_and_plot(axes[1], X_moons, y_moons, kernel='rbf', C=1.0, gamma=3.0, title="RBF • C=1 • γ=3.0")

plt.tight_layout(); plt.show()


> Nota rápida sobre avaliação
- As acurácias exibidas nos títulos são do **conjunto de treino**.
- Para medir generalização, use validação cruzada ou um conjunto de teste separado.
- Um modelo que acerta muito no treino com margens estreitas pode não generalizar bem.


# 📊 Dataset Palmer Penguins (continuação...)

In [ ]:
df = df_penguins.copy()

In [ ]:
X = df.drop(columns=['species']).copy()
y = df['species'].copy()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

In [ ]:
pre_svm = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical)
    ]
)

## Treinamento do modelo `SVC`


- `C` controla a penalidade por erros. Maior C tende a fronteiras mais ajustadas ao treino.
- `gamma` controla o alcance da curvatura no RBF. Valores altos tornam a fronteira mais ondulada.
- O GridSearch relata a melhor combinação pela acurácia média em validação cruzada.


> **🎯 Momento de Praticar**

In [ ]:
# SVC - Support Vector Classifier

pipe_svc = Pipeline(steps=[
    ('pre', pre_svm),

    # ('svm', SVC(kernel='rbf', C=1.0))  # argumentos default explícitos
    ('svm', SVC(kernel='linear'))
])

In [ ]:
pipe_svc.fit(X_train, y_train)

In [ ]:
y_pred = pipe_svc.predict(X_test)

## Avaliação dos resultados

In [ ]:
print(classification_report(y_test, y_pred, target_names=pipe_svc.classes_))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=pipe_svc.classes_)

disp.plot(cmap='Blues')
plt.title("Matriz de Confusão")
plt.tight_layout()
plt.show()

In [ ]:
cv_acc_log = cross_val_score(pipe_svc, X, y, cv=10, scoring='accuracy')

print(f"Acurácias da Validação Cruzada: {cv_acc_log}")
print(f"Acurácia Média CV: {np.mean(cv_acc_log):.4f}")
print(f"Desvio Padrão CV: {np.std(cv_acc_log):.4f}")

In [ ]:
cv_f1_log = cross_val_score(pipe_svc, X, y, cv=10, scoring='f1_macro')

print(f"F1 da Validação Cruzada: {cv_f1_log}")
print(f"F1 Média CV: {np.mean(cv_f1_log):.4f}")
print(f"Desvio Padrão CV: {np.std(cv_f1_log):.4f}")

# **🔎 GridSearch - Busca de parâmetros**

- Procura a melhor combinação de hiperparâmetros testando várias opções com validação cruzada.
- Você define um `param_grid` e uma métrica. O GridSearch treina e avalia cada combinação.
- No final, ele reaprende o modelo com os melhores parâmetros no conjunto de treino.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

> **🎯 Momento de Praticar**

In [ ]:
# prefixo deve ser o nome do passo no Pipeline
param_grid = [
    {"svm__kernel": ["linear"], "svm__C": [0.1, 1, 10]},
    {"svm__kernel": ["rbf"],    "svm__C": [0.1, 1, 10],  "svm__gamma": ["scale", 0.1, 1.0]},
]

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42) # É só o gerador de splits. Ele não treina nada.
cv

In [ ]:
gs = GridSearchCV(
    estimator=pipe_svc,
    param_grid=param_grid,
    scoring="accuracy",
    cv=cv,
)
gs

In [ ]:
gs.fit(X_train, y_train)
print("Melhores parâmetros:", gs.best_params_)
print(f"Melhor média de acurácia na CV: {gs.best_score_:.4f}")

In [ ]:
best = gs.best_estimator_
y_train_pred = best.predict(X_train)
print(f"Acurácia no teste: {accuracy_score(y_train, y_train_pred):.3f}")
print(f"F1 macro no teste: {f1_score(y_train, y_train_pred, average='macro'):.3f}")

In [ ]:
res = pd.DataFrame(gs.cv_results_)
res.head(10).sort_values("rank_test_score")

In [ ]:
cols = ["param_svm__kernel", "param_svm__C","param_svm__gamma","mean_test_score","rank_test_score"]
res[cols].sort_values("rank_test_score")

In [ ]:
# Matriz de confusão opcional
fig, ax = plt.subplots(figsize=(5,4))
ConfusionMatrixDisplay.from_estimator(best, X_train, y_train, normalize="true", cmap="Blues", ax=ax)
ax.set_title("SVC RBF no conjunto de teste")
plt.tight_layout(); plt.show()

# **💻 Exercício - Dataset Wine**

Neste exercício, você utilizará o dataset **Wine**, disponível na biblioteca `sklearn`, para treinar e avaliar modelos de **classificação supervisionada**. O conjunto de dados contém informações químicas sobre diferentes tipos de vinhos, classificados em três categorias.

Seu objetivo é conduzir todo o pipeline de Machine Learning: desde a **análise exploratória de dados** até a **escolha do melhor algoritmo de classificação** dentre as opções:  
- **K-Nearest Neighbors (KNN)**  
- **Gaussian Naive Bayes (GNB)**  
- **Support Vector Machine (SVM)**


🧭 **Etapas obrigatórias**

1. **Carregamento dos dados**
   - Use `sklearn.datasets.load_wine()` e converta os dados para um `DataFrame` com o nome `df`.
   - Adicione a coluna `'target'` com os rótulos da classe.

2. **Análise Exploratória de Dados**
   - Verifique valores ausentes (se houver).
   - Descreva estatisticamente os dados.
   - Visualize a distribuição das features e das classes.
   - Explore correlações entre variáveis e possíveis padrões.

3. **Pré-processamento**
   - Separe os dados em **conjunto de treino e teste** (ex: 70% treino / 30% teste).
   - Normalize ou padronize os dados, se necessário.
   - Utilize pipelines para as etapas de pré-processamento.

4. **Treinamento dos Modelos**
   - Treine e avalie os três algoritmos: **KNN**, **GNB** e **SVM**.
   - Utilize métricas como **acurácia**, **matriz de confusão** e **relatório de classificação**.

5. **Escolha do Melhor Modelo**
   - Compare o desempenho dos modelos com base nas métricas obtidas.
   - Escolha o algoritmo que apresenta o melhor desempenho **e justifique sua escolha**.

> **🎯 Momento de Praticar**

### **Carregamento dos Dados**

In [ ]:
import seaborn as sns
from sklearn.neighbors import KNeighborsClassifier

from sklearn.datasets import load_wine

In [ ]:
data = load_wine()

df = pd.DataFrame(data.data, columns=data.feature_names)

df['target'] = data.target

In [ ]:
df.head()

### **Análise Exploratória de Dados**

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.nunique()

In [ ]:
df.isnull().sum()

In [ ]:
df[df.duplicated(keep=False)]

In [ ]:
sns.pairplot(df, hue='target')

In [ ]:
def plot_correlation_matrix(df):
    corr_df = df[df.select_dtypes(include=['float']).columns]  # Apenas atributos contínuos
    plt.figure(figsize=(10, 8))
    correlation_matrix = corr_df.corr()
    sns.heatmap(correlation_matrix, annot=True, cmap='Blues', fmt='.2f', linewidths=0.5)
    plt.title('Matriz de Correlação')
    plt.show()

In [ ]:
plot_correlation_matrix(df)

## Pré-processamento



In [ ]:
X = df.drop(columns=['target']).copy()
y = df['target'].copy()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

In [ ]:
pre = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), X_train.columns),
    ]
)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42) # É só o gerador de splits. Ele não treina nada.
cv

## Treinamento dos modelos

### KNN

In [ ]:
pipe_knn = Pipeline(steps=[
    ('pre', pre),
    ('knn', KNeighborsClassifier())
])

In [ ]:
param_grid_knn = {
    "knn__n_neighbors": [3, 5, 7, 9],
    "knn__weights": ["uniform", "distance"],
    "knn__metric": ["euclidean", "manhattan", "chebyshev"]
}

In [ ]:
gs_knn = GridSearchCV(estimator=pipe_knn, param_grid=param_grid_knn, scoring="accuracy", cv=cv)
gs_knn.fit(X_train, y_train)

print("Melhores parâmetros:", gs_knn.best_params_)
print(f"Melhor média de acurácia na CV: {gs_knn.best_score_:.4f}")

### Naive Bayes

In [ ]:
pipe_gnb = Pipeline(steps=[
    ('pre', pre),
    ('gnb', GaussianNB())
])

In [ ]:
param_grid_gnb = {
    "gnb__var_smoothing": [1e-9, 1e-8, 1e-7, 1e-6, 1e-5]
}


In [ ]:
gs_gnb = GridSearchCV(estimator=pipe_gnb, param_grid=param_grid_gnb, scoring="accuracy", cv=cv)
gs_gnb.fit(X_train, y_train)

print("Melhores parâmetros:", gs_gnb.best_params_)
print(f"Melhor média de acurácia na CV: {gs_gnb.best_score_:.4f}")

### SVM

In [ ]:
pipe_svc = Pipeline(steps=[
    ('pre', pre),
    ('svc', SVC())
])

In [ ]:
param_grid_svc = {
    "svc__C": [0.1, 1, 10],
    "svc__kernel": ["linear", "rbf", "poly"],
    "svc__gamma": ["scale", "auto"]
}

In [ ]:
gs_svc = GridSearchCV(estimator=pipe_svc, param_grid=param_grid_svc, scoring="accuracy", cv=cv)
gs_svc.fit(X_train, y_train)

print("Melhores parâmetros:", gs_svc.best_params_)
print(f"Melhor média de acurácia na CV: {gs_svc.best_score_:.4f}")

## Comparativo

In [ ]:
pipe_knn = Pipeline(steps=[
    ('pre', pre),
    ('knn', KNeighborsClassifier(n_neighbors=5, metric='manhattan', weights='uniform'))
])

In [ ]:
pipe_gnb = Pipeline(steps=[
    ('pre', pre),
    ('gnb', GaussianNB(var_smoothing=1e-09))
])

In [ ]:
pipe_svc = Pipeline(steps=[
    ('pre', pre),
    ('svc', SVC(kernel='linear', C=0.1, gamma='scale'))
])

In [ ]:
cv_acc_knn = cross_val_score(pipe_knn, X, y, cv=10, scoring='accuracy')
print(f"Acurácia Média: {np.mean(cv_acc_knn):.4f}")

In [ ]:
cv_acc_gnb = cross_val_score(pipe_gnb, X, y, cv=10, scoring='accuracy')
print(f"Acurácia Média: {np.mean(cv_acc_gnb):.4f}")

In [ ]:
cv_acc_svc = cross_val_score(pipe_svc, X, y, cv=10, scoring='accuracy')
print(f"Acurácia Média: {np.mean(cv_acc_svc):.4f}")